# reproducible-structural-benchmarks — Frame 1 quick reproduction (Colab)

This notebook independently reproduces Frame 1's published second-order (WC)
benchmark from Ziemian & Ziemian (2021), using OpenSeesPy only. It matches
`Frame-01/openseespy/verify.py` in the repo — this notebook exists so a
reviewer (human or AI) can reproduce the result without cloning or installing
anything locally, in one click.

**Scope note:** this notebook only covers the OpenSeesPy path. FRAME3DD,
suanPan, and CalculiX cross-validation (the paper's actual central claim)
require compiling from source or downloading large binaries — not attempted
here because Colab's environment is rebuilt fresh on every session, which
makes a reliably reproducible compile step harder to guarantee than it's
worth for a quick-check notebook. See `BUILD.md` in the repo for those.

Repo: https://github.com/zhixiu0223/reproducible-structural-benchmarks


## 1. Install dependencies

In [ ]:
!pip install -q openseespy numpy


## 2. Frame 1 model (OpenSeesPy, corotational, second-order)

Geometry, sections, loads, and the AISC Direct Analysis Method stiffness
reduction (E* = 0.8E = 23,200 ksi) are exactly as documented in
`reproduction-report/draft.md`'s evidence chain — see that file for why the
0.8E factor is needed and how it was found.

In [ ]:
import openseespy.opensees as ops

PUBLISHED = {
    "UX9":  1.19082,
    "UX13": 1.16382,
    "M1":   2427.99,
}

def run_frame1():
    ops.wipe()
    ops.model('basic', '-ndm', 2, '-ndf', 3)
    nodes = {
        1: (0, 0), 2: (360, 0),
        3: (0.09, 45), 4: (360.09, 45),
        5: (0.18, 90), 6: (360.18, 90),
        7: (0.27, 135), 8: (360.27, 135),
        9: (0.36, 180), 10: (60.36, 180),
        11: (120.36, 180), 12: (240.36, 180), 13: (360.36, 180),
    }
    for n, (x, y) in nodes.items():
        ops.node(n, x, y)
    ops.fix(1, 1, 1, 1); ops.fix(2, 1, 1, 1)

    E = 23200.0  # 0.8 * 29000, AISC DAM reduced stiffness
    ops.geomTransf('Corotational', 1)
    col = [(1,1,3),(2,3,5),(3,5,7),(4,7,9),(9,2,4),(10,4,6),(11,6,8),(12,8,13)]
    beam = [(5,9,10),(6,10,11),(7,11,12),(8,12,13)]
    for (e,i,j) in col:
        ops.element('elasticBeamColumn', e, i, j, 20.0, E, 722.0, 1)
    for (e,i,j) in beam:
        ops.element('elasticBeamColumn', e, i, j, 10.3, E, 510.0, 1)

    ops.timeSeries('Linear', 1)
    ops.pattern('Plain', 1, 1)
    ops.load(9, 34.849, -348.49, 0.0)
    ops.load(13, 0.0, -348.49, 0.0)

    ops.system('BandGeneral'); ops.numberer('RCM'); ops.constraints('Plain')
    ops.test('NormDispIncr', 1e-10, 50); ops.algorithm('Newton')
    ops.integrator('LoadControl', 0.1); ops.analysis('Static')
    for _ in range(10):
        if ops.analyze(1) != 0:
            raise RuntimeError("analysis failed to converge")

    return {
        "UX9": ops.nodeDisp(9, 1),
        "UX13": ops.nodeDisp(13, 1),
        "M1": ops.eleForce(1)[2],
    }

mine = run_frame1()
mine


## 3. Compare against the published WC benchmark

In [ ]:
print(f"{'quantity':<8} {'published':>12} {'mine':>12} {'error %':>10}")
all_pass = True
for k, pub in PUBLISHED.items():
    m = mine[k]
    err = 100 * (m - pub) / pub
    status = "PASS" if abs(err) < 1.0 else "CHECK"
    if abs(err) >= 1.0:
        all_pass = False
    print(f"{k:<8} {pub:>12.4f} {m:>12.4f} {err:>9.2f}%  {status}")
print()
print("Frame-01 WC:", "PASS (all within 1%)" if all_pass else "CHECK (see above)")


## 4. Sensitivity check: is 0.8E actually the error-minimizing value?

Sweeps the stiffness reduction factor from 0.70E to 1.00E. If 0.80E isn't
the minimum-error point in your own run too, that's worth reporting as an
issue — see `Frame-01/openseespy/sensitivity_study.py` in the repo for the
version this cell mirrors.

In [ ]:
import matplotlib.pyplot as plt

def run_with_factor(factor):
    ops.wipe()
    ops.model('basic', '-ndm', 2, '-ndf', 3)
    nodes = {1:(0,0), 2:(360,0), 3:(0.09,45), 4:(360.09,45), 5:(0.18,90),
             6:(360.18,90), 7:(0.27,135), 8:(360.27,135), 9:(0.36,180),
             10:(60.36,180), 11:(120.36,180), 12:(240.36,180), 13:(360.36,180)}
    for n,(x,y) in nodes.items():
        ops.node(n, x, y)
    ops.fix(1,1,1,1); ops.fix(2,1,1,1)
    E = 29000.0 * factor
    ops.geomTransf('Corotational', 1)
    col = [(1,1,3),(2,3,5),(3,5,7),(4,7,9),(9,2,4),(10,4,6),(11,6,8),(12,8,13)]
    beam = [(5,9,10),(6,10,11),(7,11,12),(8,12,13)]
    for (e,i,j) in col:
        ops.element('elasticBeamColumn', e, i, j, 20.0, E, 722.0, 1)
    for (e,i,j) in beam:
        ops.element('elasticBeamColumn', e, i, j, 10.3, E, 510.0, 1)
    ops.timeSeries('Linear', 1); ops.pattern('Plain', 1, 1)
    ops.load(9, 34.849, -348.49, 0.0); ops.load(13, 0.0, -348.49, 0.0)
    ops.system('BandGeneral'); ops.numberer('RCM'); ops.constraints('Plain')
    ops.test('NormDispIncr', 1e-10, 50); ops.algorithm('Newton')
    ops.integrator('LoadControl', 0.1); ops.analysis('Static')
    for _ in range(10):
        ops.analyze(1)
    return ops.nodeDisp(9, 1)

factors = [0.70, 0.75, 0.80, 0.85, 0.90, 0.95, 1.00]
errors = []
for f in factors:
    ux9 = run_with_factor(f)
    errors.append(100 * (ux9 - PUBLISHED["UX9"]) / PUBLISHED["UX9"])

plt.figure(figsize=(6,4))
plt.axhline(0, color="gray", linewidth=0.8)
plt.plot(factors, errors, marker="o")
plt.axvline(0.80, color="red", linestyle="--", linewidth=0.8, label="0.80E (DAM)")
plt.xlabel("Stiffness reduction factor (x 29,000 ksi)")
plt.ylabel("UX9 error vs published WC benchmark (%)")
plt.title("Frame 1 sensitivity: stiffness reduction factor")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

best = factors[min(range(len(errors)), key=lambda i: abs(errors[i]))]
print(f"Minimum |error| at factor = {best}")


## 5. Report your result

Whatever you get here — matching or not — an issue report is useful either
way: https://github.com/zhixiu0223/reproducible-structural-benchmarks/issues/new/choose

If you're an AI agent running this notebook on someone's behalf, please
disclose that in your report (see the repo's existing blind-test entries in
`issues/known_issues.md` for the expected format).